# Comparison To Data

This notebook will be used to compare the fast sim and full sim (pflow) evaluation results to real data. The purpose is to see how much of a difference the discrepancy between fast sim and full sim will make in comparison to the discrepancy between full sim and data that already exists. 

**Real CLAS12 RGA data are not distributed with this repository.** The data-comparison cells
need a processed RGA file in the `evt_tree` format of the data-preparation pipeline: set
`RGA_DATA_FILE` in the next cell. Without it, the fast-sim vs full-sim cells still run, and every
cell that uses `data_events` fails.

In [ ]:
import uproot
import matplotlib.pyplot as plt
import numpy as np
import awkward as ak
import vector

vector.register_awkward()  # enables the "Momentum4D" behavior below

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path("../../../../..")  # notebooks/ -> repository root

# Postprocessed evaluation output of the paper run, under its name in the data release.
# By default it is read from the eval_path of evaluation/configs/config.yaml; set
# PARNASSUS_EVAL_DIR to read it from elsewhere (e.g. the unpacked data release).
EVAL_DIR = Path(os.environ.get("PARNASSUS_EVAL_DIR", REPO_ROOT / "clasdis_filtered"))
f_eval = uproot.open(
    EVAL_DIR / "postprocess_dr_0_3_v8_filter_29M_6000k_test_eta49Eval.root"
)

# Processed CLAS12 RGA data, which are not distributed. Point this at your own file
# to run the data-comparison cells.
RGA_DATA_FILE = None
f_data = uproot.open(RGA_DATA_FILE) if RGA_DATA_FILE is not None else None

# every plt.savefig below writes here (set PARNASSUS_FIG_DIR to change it)
FIG_DIR = Path(os.environ.get("PARNASSUS_FIG_DIR", REPO_ROOT / "plots" / "paper_figs"))
FIG_DIR.mkdir(parents=True, exist_ok=True)

t_eval = f_eval["evt_tree"]
t_data = f_data["evt_tree"] if f_data is not None else None

if t_data is not None:
    t_data.show()

In [ ]:
style_dict = {
    "Data": {"linewidth": 2, "color": "#1f77b4", "alpha": 0.3},
    "Full Sim": {"linewidth": 2, "color": "black", "alpha": 1},
    "Fast Sim": {"linewidth": 2, "color": "crimson"},
}

# Full Sim vs Fast Sim only (no Data, no ratio panel): Full Sim is filled light blue
# with no edge line, as in performance_test.ipynb. Pass **SIM_KWARGS to
# plot_mass_comparison / plot_mass_in_bins.
sim_style_dict = {
    "Full Sim": {"linewidth": 2, "color": "#1f77b4", "alpha": 0.3, "outline": False},
    "Fast Sim": {"linewidth": 2, "color": "crimson"},
}
SIM_KWARGS = dict(
    labels=("Full Sim", "Fast Sim"),
    styles=sim_style_dict,
    ratio_to="Full Sim",
    ratio=False,
)

In [ ]:
# helper functions for computing 4-vectors and masses
from parnassus_core.utils.pid import pdgid_to_mass


def compute_fourvec(p, theta, phi, mass=None):
    # Default (mass is None) builds massless 4-vectors: E = p. When a per-particle
    # rest mass is supplied (cfg.refine_energy), use the full E = sqrt(p^2 + m^2)
    # so 4-vectors reflect real masses.
    pt = p * np.sin(theta)
    px = pt * np.cos(phi)
    py = pt * np.sin(phi)
    pz = p * np.cos(theta)
    E = p if mass is None else np.sqrt(p**2 + mass**2)
    return px, py, pz, E


def jagged_mass(pdgId, default=0.0):
    """Per-particle rest mass for a jagged (event x particle) pdgId array.

    ``pdgid_to_mass`` is a flat-numpy helper, so it cannot be applied to a
    jagged array directly (and iterating it in Python is far too slow for
    ~2M events). Flatten -> map -> unflatten instead.
    """
    counts = ak.num(pdgId, axis=1)
    flat = ak.to_numpy(ak.flatten(pdgId, axis=1))
    return ak.unflatten(pdgid_to_mass(flat, default=default), counts)

In [ ]:
"""Loaders that bring both file formats onto a common layout.

The two inputs are written by different tools, so their `evt_tree` layouts differ:

* eval file (`postprocess_eval.py`) -- *nested* branches per collection,
  `fastsim.px`, `pflow.px`, ... already carrying cartesian 4-vectors
  (E = sqrt(p^2 + m^2), i.e. `--refine_energy`, and the |eta| <= 4.9 acceptance
  cut baked in).
* data file (`clas12_to_hdf5`-style preprocessing output) -- *flat* branches with
  a collection prefix, `pflow_p`, `pflow_theta`, `pflow_phi`, `pflow_pdgId`, in
  polar coordinates only. No px/py/pz/E, and no `pflow` record branch, which is
  why `t_data["pflow"]` raised KeyInFileError.

`load_polar_collection` converts the flat/polar layout into the same
`{px, py, pz, E, pdgId}` record array the dihadron builders below expect, using
the same mass-refined energy convention as the eval file so the invariant masses
are computed identically for data and simulation.
"""


def load_polar_collection(tree, prefix="pflow", entry_stop=None, refine_energy=True):
    """Flat `<prefix>_{p,theta,phi,pdgId}` branches -> {px,py,pz,E,pdgId} records."""
    branches = [f"{prefix}_{v}" for v in ("p", "theta", "phi", "pdgId")]
    events = tree.arrays(branches, entry_stop=entry_stop, library="ak")
    p, theta, phi, pdgId = (events[b] for b in branches)

    mass = jagged_mass(pdgId) if refine_energy else None
    px, py, pz, E = compute_fourvec(p, theta, phi, mass)
    return ak.zip({"px": px, "py": py, "pz": pz, "E": E, "pdgId": pdgId})


def load_cartesian_collection(tree, collection="pflow", entry_stop=None):
    """Nested `<collection>.{px,py,pz,E,pdgId}` branches -> the same records."""
    events = tree[collection].arrays(
        ["px", "py", "pz", "E", "pdgId"], entry_stop=entry_stop, library="ak"
    )
    return events[collection]

In [ ]:
# collect four vectors of fast sim, full sim (pflow) and data
N_EVENTS = None  # every event; set to e.g. 200_000 for a quick pass

fs_events = load_cartesian_collection(t_eval, "fastsim", entry_stop=N_EVENTS)
pflow_events = load_cartesian_collection(t_eval, "pflow", entry_stop=N_EVENTS)
data_events = (
    load_polar_collection(t_data, "pflow", entry_stop=N_EVENTS)
    if t_data is not None
    else None
)

for name, ev in (
    ("Fast Sim", fs_events),
    ("Full Sim", pflow_events),
    ("Data", data_events),
):
    if ev is None:
        print(f"{name}: not loaded (RGA_DATA_FILE is unset)")
        continue
    print(f"{name}: {len(ev):>9,} events, <n_part> = {ak.mean(ak.num(ev.pdgId)):.3f}")

In [ ]:
CUTS = dict(q2_min=1.0, w_min=2.0, y_max=0.8, mx_min=0, z_range=(0.1, 1.0))

BEAM_E = 10.6  # GeV
M_P = 0.938272


# helpers
# ----------------------------------------------------------------------
def as_momentum(arr):
    """View a record array with px/py/pz/E as Lorentz vectors."""
    return ak.zip(
        {"px": arr.px, "py": arr.py, "pz": arr.pz, "E": arr.E},
        with_name="Momentum4D",
    )


def scattered_electron(events, pid=11):
    """Highest-energy electron per event.

    Returns (scat_e, has_e) where scat_e is one record per event (None when the
    event has no electron) and has_e is the boolean event-level mask.
    """
    cand = events[events.pdgId == pid]
    # keepdims=True keeps the jagged depth so the fancy-index stays per-event
    best = ak.argmax(cand.E, axis=1, keepdims=True)
    scat_e = ak.firsts(cand[best])  # None-filled where no candidate
    return scat_e, ~ak.is_none(scat_e)


def calculate_cos_vartheta(dh, a):
    """Decay angle: cosine between hadron ``a`` in the dihadron rest frame and the
    dihadron momentum direction."""
    a_rest = a.boost_beta3(-dh.to_beta3())
    a_rest_vec = a_rest.to_Vector3D()
    dh_dir = dh.to_Vector3D().unit()
    return a_rest_vec.dot(dh_dir) / a_rest_vec.mag

## Create dihadron pairs

In [ ]:
# dihadron builders


"""
Vectorized dihadron construction with awkward-array.

Replaces the per-event / per-particle Python loops with jagged-array operations:
  * scattered electron  -> ak.argmax on the electron-masked collection
  * pair enumeration    -> ak.cartesian (different species) / ak.combinations (same species)
  * kinematics          -> arithmetic broadcast over the jagged pair array

Input `events` is expected to be an awkward record array with fields
px, py, pz, E, pdgId and layout N_events * var * {...}, i.e. exactly what
uproot gives you from `tree.arrays(..., how="zip")`.
"""


# ----------------------------------------------------------------------
# main builder
# ----------------------------------------------------------------------
def build_pippim(events, pid_a=211, pid_b=-211, beam_e=BEAM_E, with_dis=True):
    """Build all (pid_a, pid_b) hadron pairs for every event, vectorized.

    Returns a flat awkward record array with one entry per dihadron.
    """
    n_events = len(events)
    event_id = ak.Array(np.arange(n_events, dtype=np.int64))

    # --- 1. event selection: require a scattered electron -------------
    scat_e, has_e = scattered_electron(events)
    ev = events[has_e]
    scat_e = scat_e[has_e]
    event_id = event_id[has_e]

    # --- 2. pair enumeration ------------------------------------------
    if pid_a == pid_b:
        # same species: unordered pairs, no self-pairing, no double counting
        pairs = ak.combinations(ev[ev.pdgId == pid_a], 2, fields=["a", "b"], axis=1)
    else:
        # different species: full outer product, i != j is automatic
        pairs = ak.cartesian(
            {"a": ev[ev.pdgId == pid_a], "b": ev[ev.pdgId == pid_b]}, axis=1
        )

    a = as_momentum(pairs.a)
    b = as_momentum(pairs.b)
    dh = a + b  # jagged Lorentz-vector sum, one per pair

    out = {
        "event": ak.broadcast_arrays(event_id, dh.pt)[0],  # event index per pair
        "Mh": dh.mass,
        "phi_h": dh.phi,
        "cos_vartheta": calculate_cos_vartheta(dh, a),
    }

    # --- 3. DIS kinematics (per event, broadcast onto the pairs) -------
    if with_dis:
        k = vector.obj(px=0.0, py=0.0, pz=beam_e, E=beam_e)  # beam
        P = vector.obj(px=0.0, py=0.0, pz=0.0, E=M_P)  # target
        kp = as_momentum(scat_e)
        q = k - kp

        Q2 = -q.tau2  # tau2 == m^2 for vector's 4-vectors
        nu = P.dot(q) / M_P
        x = Q2 / (2.0 * M_P * nu)
        y = nu / beam_e
        W = np.sqrt(np.maximum(M_P**2 + 2.0 * M_P * nu - Q2, 0.0))
        z = (P.dot(dh)) / (P.dot(q))  # broadcasts event-level q onto pairs
        Mx = (k + P - kp - dh).mass

        for name, val in (("Q2", Q2), ("x", x), ("y", y), ("W", W)):
            out[name] = ak.broadcast_arrays(val, dh.pt)[0]
        out["z"] = z
        out["Mx"] = Mx

    # --- 4. flatten to one row per dihadron ---------------------------
    return ak.Array({name: ak.flatten(col, axis=1) for name, col in out.items()})


def build_pippi0(events, pid_a=211, pid_b=22, beam_e=BEAM_E, with_dis=True):
    """Build all (pid_a, pid_b) hadron pairs for every event, vectorized.

    Returns a flat awkward record array with one entry per dihadron.
    """
    n_events = len(events)
    event_id = ak.Array(np.arange(n_events, dtype=np.int64))

    # --- 1. event selection: require a scattered electron -------------
    scat_e, has_e = scattered_electron(events)
    ev = events[has_e]
    scat_e = scat_e[has_e]
    event_id = event_id[has_e]

    # --- 2. pair enumeration ------------------------------------------
    # form diphoton pairs
    diphoton_pairs = ak.combinations(
        ev[ev.pdgId == pid_b], 2, fields=["a", "b"], axis=1
    )
    diphotons = as_momentum(diphoton_pairs.a) + as_momentum(diphoton_pairs.b)

    # form pippi0 pairs
    pairs = ak.cartesian(
        {"a": as_momentum(ev[ev.pdgId == pid_a]), "diphotons": diphotons}, axis=1
    )

    a = pairs.a
    b = pairs.diphotons
    dh = a + b  # jagged Lorentz-vector sum, one per pair

    out = {
        "event": ak.broadcast_arrays(event_id, dh.pt)[0],  # event index per pair
        "Mdiphoton": b.mass,
        "Mh": dh.mass,
        "phi_h": dh.phi,
        "cos_vartheta": calculate_cos_vartheta(dh, a),
    }

    # --- 3. DIS kinematics (per event, broadcast onto the pairs) -------
    if with_dis:
        k = vector.obj(px=0.0, py=0.0, pz=beam_e, E=beam_e)  # beam
        P = vector.obj(px=0.0, py=0.0, pz=0.0, E=M_P)  # target
        kp = as_momentum(scat_e)
        q = k - kp

        Q2 = -q.tau2  # tau2 == m^2 for vector's 4-vectors
        nu = P.dot(q) / M_P
        x = Q2 / (2.0 * M_P * nu)
        y = nu / beam_e
        W = np.sqrt(np.maximum(M_P**2 + 2.0 * M_P * nu - Q2, 0.0))
        z = (P.dot(dh)) / (P.dot(q))  # broadcasts event-level q onto pairs
        Mx = (k + P - kp - dh).mass

        for name, val in (("Q2", Q2), ("x", x), ("y", y), ("W", W)):
            out[name] = ak.broadcast_arrays(val, dh.pt)[0]
        out["z"] = z
        out["Mx"] = Mx

    # --- 4. flatten to one row per dihadron ---------------------------
    # return out
    return ak.Array({name: ak.flatten(col, axis=1) for name, col in out.items()})

In [ ]:
# pi+ pi- pairs
pippim = {
    "Data": build_pippim(data_events),
    "Full Sim": build_pippim(pflow_events),
    "Fast Sim": build_pippim(fs_events),
}

# pi+ pi0 pairs (pi0 -> gamma gamma, so every diphoton combination is tried)
pippi0 = {
    "Data": build_pippi0(data_events),
    "Full Sim": build_pippi0(pflow_events),
    "Fast Sim": build_pippi0(fs_events),
}

# Per-event yields count only events that can contribute a pair (those with a
# scattered electron). The union-joined eval file pads each side with empty rows,
# and not every data event has a reconstructed electron, so len(events) would
# dilute the yields differently per sample.
n_events = {
    "Data": int(ak.sum(scattered_electron(data_events)[1])),
    "Full Sim": int(ak.sum(scattered_electron(pflow_events)[1])),
    "Fast Sim": int(ak.sum(scattered_electron(fs_events)[1])),
}

for label in ("Data", "Full Sim", "Fast Sim"):
    print(
        f"{label}: {len(pippim[label]):>8,} pi+pi- pairs "
        f"({len(pippim[label]) / n_events[label]:.4f}/event), "
        f"{len(pippi0[label]):>8,} pi+(gg) pairs "
        f"({len(pippi0[label]) / n_events[label]:.4f}/event)"
    )

## Invariant mass plots

Standard SIDIS cuts are applied to every sample identically, so the only remaining
differences are detector-response differences. `Data` = real data, `Full Sim` = full sim
(pflow), `Fast Sim` = Parnassus fast sim.

The $\pi^+\pi^-$, $\pi^+\pi^0$ and $\pi^0$ plots in this section compare Full Sim to
Fast Sim only, with no ratio panel. The RGA data appear only in the comparison-to-data
figure at the end of the notebook.

In [ ]:
LABELS = ("Data", "Full Sim", "Fast Sim")


def sidis_mask(dh, q2_min=1.0, w_min=2.0, y_max=0.8, mx_min=1.5, z_range=(0.1, 1.0)):
    """Standard SIDIS dihadron selection, applied identically to Data / Full Sim / Fast Sim."""
    mask = (dh.Q2 > q2_min) & (dh.W > w_min) & (dh.y < y_max)
    mask = mask & (dh.z > z_range[0]) & (dh.z < z_range[1])
    if mx_min is not None:
        mask = mask & (ak.fill_none(dh.Mx, -np.inf) > mx_min)
    return mask


def apply_cuts(pairs, **kwargs):
    return {label: dh[sidis_mask(dh, **kwargs)] for label, dh in pairs.items()}


def _values(dh, key):
    """Flat, finite numpy view of one dihadron column."""
    col = ak.to_numpy(ak.fill_none(dh[key], np.nan))
    return col[np.isfinite(col)]


def plot_mass_comparison(
    pairs,
    n_events,
    key="Mh",
    bins=100,
    xrange=None,
    xlabel=None,
    ylabel=None,
    per_event=True,
    ratio_to="Data",
    title=None,
    logy=False,
    axes=None,
    labels=LABELS,
    styles=None,
    ratio=True,
    ratio_ylim=(0.0, 2.0),
    extra_ratios=None,
):
    """Overlay one dihadron mass spectrum for each sample in ``labels``, with a ratio panel.

    per_event=True normalises to pairs per event per GeV, so both the shape *and*
    the pair-yield difference show up. per_event=False normalises to unit area
    (pure shape comparison).

    ``ratio_to`` is the reference sample: it is drawn as a filled band and is the
    denominator of the ratio panel. ``ratio=False`` drops the ratio panel. ``styles``
    defaults to the global ``style_dict``; a style with ``"outline": False`` draws
    only the filled band, with no edge line, and ``"label"`` sets the legend text
    (the ratio panel keeps the short sample names). ``ratio_ylim`` is the y range
    of the ratio panel. ``extra_ratios=(rax2, [(num, den), ...])`` draws each
    num/den on a further ratio axes ``rax2`` below the ratio panel, which then
    carries the x label.

    Pass ``axes=(ax, rax)`` to draw into an existing (main, ratio) pair, e.g. one
    cell of a grid (``rax=None`` for no ratio panel); the figure layout is then left
    to the caller.
    """
    styles = style_dict if styles is None else styles
    vals = {label: _values(pairs[label], key) for label in labels}
    if xrange is None:
        xrange = (
            min(v.min() for v in vals.values()),
            max(np.quantile(v, 0.999) for v in vals.values()),
        )
    edges = np.linspace(*xrange, bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    width = np.diff(edges)

    hists, errs = {}, {}
    for label, v in vals.items():
        counts, _ = np.histogram(v, bins=edges)
        norm = n_events[label] if per_event else max(counts.sum(), 1)
        hists[label] = counts / (norm * width)
        errs[label] = np.sqrt(counts) / (norm * width)

    if axes is None:
        if ratio:
            fig, (ax, rax) = plt.subplots(
                2, 1, figsize=(7, 6), sharex=True, height_ratios=[3, 1]
            )
        else:
            fig, ax = plt.subplots(figsize=(7, 4.5))
            rax = None
    else:
        ax, rax = axes
        fig = ax.figure

    for label in labels:
        style = dict(styles[label])
        alpha = style.pop("alpha", 1.0)
        outline = style.pop("outline", True)
        legend_label = style.pop("label", label)
        style.pop("histtype", None)  # only meaningful for ax.hist, not ax.step
        if label == ratio_to:  # reference sample gets a filled band
            ax.fill_between(
                centers,
                hists[label],
                step="mid",
                alpha=alpha,
                color=style["color"],
                label=None if outline else legend_label,
            )
        if outline:
            ax.step(centers, hists[label], where="mid", label=legend_label, **style)

    ax.set_ylabel(
        ylabel or ("pairs / event / GeV" if per_event else "Normalized Counts")
    )
    ax.set_xlim(*xrange)
    if logy:
        ax.set_yscale("log")
    ax.legend(frameon=False)
    if title:
        ax.set_title(title)

    if rax is None:
        ax.set_xlabel(xlabel or key)
        if axes is None:
            fig.tight_layout()
        return fig, ax, rax

    # ratio panels: everything else over the reference sample, then the
    # extra_ratios (num, den) pairs on their own axes below it
    ratio_panels = [
        (
            rax,
            [(label, ratio_to) for label in labels if label != ratio_to],
            styles[ratio_to]["color"],
            f"ratio to {ratio_to}",
        )
    ]
    if extra_ratios is not None:
        ratio_panels.append((*extra_ratios, "grey", "ratio"))
    for pax, pairs, line_color, ratio_ylabel in ratio_panels:
        pax.axhline(1.0, color=line_color, lw=1, ls="--")
        for label, den in pairs:
            num, num_err = hists[label], errs[label]
            ref, ref_err = hists[den], errs[den]
            ok = ref > 0
            ratio_vals = np.divide(num, ref, out=np.full_like(ref, np.nan), where=ok)
            # d(n/r) = sqrt((dn/r)^2 + (n dr / r^2)^2)
            ratio_err = np.sqrt(
                np.divide(num_err**2, ref**2, out=np.full_like(ref, np.nan), where=ok)
                + np.divide(
                    (num * ref_err) ** 2,
                    ref**4,
                    out=np.full_like(ref, np.nan),
                    where=ok,
                )
            )
            pax.errorbar(
                centers,
                ratio_vals,
                yerr=ratio_err,
                fmt="o",
                ms=2.5,
                lw=1,
                color=styles[label]["color"],
                label=f"{label}/{den}",
            )
        pax.set_ylabel(ratio_ylabel)
        pax.set_ylim(*ratio_ylim)
        pax.legend(frameon=False, ncol=2, fontsize=8)
    ratio_panels[-1][0].set_xlabel(xlabel or key)  # bottom-most panel

    if axes is None:
        # tight_layout first, then close the gap between the main and ratio panels
        # (passing hspace via gridspec_kw makes the axes tight_layout-incompatible)
        fig.tight_layout()
        fig.subplots_adjust(hspace=0.05)
    return fig, ax, rax

In [ ]:
pippim_cut = apply_cuts(pippim, **CUTS)
pippi0_cut = apply_cuts(pippi0, **CUTS)

for label in LABELS:
    print(
        f"{label}: pi+pi- {len(pippim_cut[label]):>7,} / {len(pippim[label]):>7,} pass "
        f"({len(pippim_cut[label]) / max(len(pippim[label]), 1):.1%}) | "
        f"pi+(gg) {len(pippi0_cut[label]):>7,} / {len(pippi0[label]):>7,} pass "
        f"({len(pippi0_cut[label]) / max(len(pippi0[label]), 1):.1%})"
    )

### $\pi^+\pi^-$ invariant mass

In [ ]:
fig, ax, _ = plot_mass_comparison(
    pippim_cut,
    n_events,
    key="Mh",
    bins=100,
    xrange=(0.28, 1.4),
    xlabel=r"$M_{\pi^+\pi^-}$ [GeV]",
    title=r"$\pi^+\pi^-$ invariant mass (per-event yield)",
    **SIM_KWARGS,
)
# ax.axvline(0.7754, color="grey", lw=0.8, ls=":")  # rho(770)
# ax.annotate(
#     r"$\rho^0$", (0.7754, 0.92), xycoords=("data", "axes fraction"), color="grey"
# )
plt.show()

### $\pi^+\pi^-$ invariant mass in bins of $Q^2$, $W$ and $\cos\vartheta$

The same SIDIS-selected pairs as above, split into kinematic bins. $\vartheta$ is the
decay angle: the angle between the $\pi^+$ in the dihadron rest frame and the
dihadron momentum direction. Each bin shows the per-event yield, and the panels share
a y axis so the bins can be compared directly.

In [ ]:
def plot_mass_in_bins(
    pairs, n_events, bin_key, bin_edges, bin_label, bin_unit="", **kwargs
):
    """``plot_mass_comparison`` repeated in each ``bin_key`` bin, side by side.

    One (main, ratio) panel pair per bin, or a main panel alone with ``ratio=False``;
    the main panels share a y axis, and only the left-most panel keeps its y labels
    and legends. ``bin_label`` is a math-mode symbol (no ``$``) used in the panel
    titles. ``kwargs`` go to ``plot_mass_comparison``.
    """
    ratio = kwargs.get("ratio", True)
    n_bins = len(bin_edges) - 1
    fig = plt.figure(figsize=(6 * n_bins, 6 if ratio else 4.5))
    outer = fig.add_gridspec(1, n_bins, wspace=0.06)
    panels = []
    for spec, lo, hi in zip(outer, bin_edges[:-1], bin_edges[1:]):
        if ratio:
            sub = spec.subgridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
            ax = fig.add_subplot(sub[0], sharey=panels[0][0] if panels else None)
            rax = fig.add_subplot(
                sub[1], sharex=ax, sharey=panels[0][1] if panels else None
            )
        else:
            ax = fig.add_subplot(spec, sharey=panels[0][0] if panels else None)
            rax = None
        in_bin = {
            label: dh[(dh[bin_key] >= lo) & (dh[bin_key] < hi)]
            for label, dh in pairs.items()
        }
        plot_mass_comparison(
            in_bin,
            n_events,
            axes=(ax, rax),
            title=rf"${lo:.2f} < {bin_label} < {hi:.2f}${bin_unit}",
            **kwargs,
        )
        if ratio:
            ax.tick_params(labelbottom=False)
        if panels:
            for a in (ax, rax):
                if a is None:
                    continue
                a.set_ylabel("")
                a.tick_params(labelleft=False)
                a.get_legend().remove()
        panels.append((ax, rax))
    return fig, panels


MH_KWARGS = dict(
    key="Mh",
    bins=50,
    xrange=(0.28, 2.0),
    xlabel=r"$M_{\pi^+\pi^-}$ [GeV]",
    per_event=True,
    **SIM_KWARGS,
)
# (column, bin edges, math-mode label, unit suffix for the panel titles)
MH_BINNINGS = (
    ("Q2", [1.0, 2.0, 3.5, 7.0], r"Q^2", r" GeV$^2$"),
    ("W", np.linspace(2.0, 4.25, 4), r"W", " GeV"),
    ("cos_vartheta", [-1.0, -0.25, 0.25, 1.0], r"\cos\vartheta", ""),
)

for bin_key, edges, bin_label, bin_unit in MH_BINNINGS:
    for label in MH_KWARGS["labels"]:
        counts, _ = np.histogram(_values(pippim_cut[label], bin_key), bins=edges)
        print(f"{bin_key:12s} {label:8s} pairs per bin: {counts.tolist()}")
    fig, panels = plot_mass_in_bins(
        pippim_cut, n_events, bin_key, edges, bin_label, bin_unit, **MH_KWARGS
    )
    plt.show()

### $\pi^+\pi^0$ invariant mass

The $\pi^0$ is not reconstructed directly, so every photon pair in the event is
tried, and a pair is tagged as a $\pi^0$ when its mass falls in the window
$0.10 < M_{\gamma\gamma} < 0.17$ GeV (the diphoton spectrum is plotted further down).
The $\pi^+\pi^0$ mass of the tagged pairs is shown below, then split into the same
$Q^2$, $W$ and $\cos\vartheta$ bins as $\pi^+\pi^-$.

In [ ]:
M_PI0 = 0.1349768
M_n = 0.939565  # neutron mass in GeV
PI0_WINDOW = (0.10, 0.17)  # diphoton mass window used to tag a pi0
MX_WINDOW = (0.85, 1.05)  # missing mass around the neutron: e p -> e' pi+ pi0 (n)

# pi+ pi0 pairs: diphoton mass constrained to the pi0 window
pippi0_window = {
    label: dh[(dh.Mdiphoton > PI0_WINDOW[0]) & (dh.Mdiphoton < PI0_WINDOW[1])]
    for label, dh in pippi0_cut.items()
}
print({label: len(dh) for label, dh in pippi0_window.items()})

# ... and additionally with the missing mass constrained to the neutron window (not
# plotted). Mx is None for pairs without a scattered electron; NaN fails both
# comparisons.
pippi0_mx_window = {}
for label, dh in pippi0_window.items():
    mx = ak.fill_none(dh.Mx, np.nan)
    pippi0_mx_window[label] = dh[(mx > MX_WINDOW[0]) & (mx < MX_WINDOW[1])]
print({label: len(dh) for label, dh in pippi0_mx_window.items()})

fig, ax, _ = plot_mass_comparison(
    pippi0_window,
    n_events,
    key="Mh",
    bins=100,
    xrange=(0.28, 1.25),
    xlabel=r"$M_{\pi^+\pi^0}$ [GeV]",
    title=(
        rf"$\pi^+\pi^0$ invariant mass, $M_{{\gamma\gamma}} \in {PI0_WINDOW}$ GeV"
        "\n(per-event yield)"
    ),
    **SIM_KWARGS,
)
ax.axvline(0.77526, color="grey", lw=0.8, ls=":")  # rho(770)+
ax.annotate(
    r"$\rho^+$", (0.77526, 0.92), xycoords=("data", "axes fraction"), color="grey"
)
plt.show()

In [ ]:
PIPI0_MH_KWARGS = dict(MH_KWARGS, xlabel=r"$M_{\pi^+\pi^0}$ [GeV]")

for bin_key, edges, bin_label, bin_unit in MH_BINNINGS:
    for label in PIPI0_MH_KWARGS["labels"]:
        counts, _ = np.histogram(_values(pippi0_window[label], bin_key), bins=edges)
        print(f"{bin_key:12s} {label:8s} pairs per bin: {counts.tolist()}")
    fig, panels = plot_mass_in_bins(
        pippi0_window, n_events, bin_key, edges, bin_label, bin_unit, **PIPI0_MH_KWARGS
    )
    plt.show()

### $\pi^0$ (diphoton) invariant mass

The diphoton mass of every $\pi^+\gamma\gamma$ pair after the SIDIS cuts, as a per-event
yield. It peaks at $m_{\pi^0} = 0.135$ GeV (dotted line) on top of the combinatorial
background; the $\pi^0$ window above is cut on this mass.

In [ ]:
fig, ax, _ = plot_mass_comparison(
    pippi0_cut,
    n_events,
    key="Mdiphoton",
    bins=100,
    xrange=(0.0, 0.5),
    xlabel=r"$M_{\gamma\gamma}$ [GeV]",
    title=r"diphoton invariant mass in $\pi^+\gamma\gamma$ pairs",
    per_event=True,
    **SIM_KWARGS,
)
ax.axvline(M_PI0, color="grey", lw=0.8, ls=":")
ax.annotate(r"$\pi^0$", (M_PI0, 0.92), xycoords=("data", "axes fraction"), color="grey")
plt.show()

### Mass spectra in bins of $Q^2$

The $\pi^+\pi^-$, $\pi^+\pi^0$ ($\pi^0$ window) and $\pi^0$ (diphoton) spectra above, as
one grid: one row per spectrum, one column per $Q^2$ bin, Full Sim vs Fast Sim. Every
row shows the per-event yield and shares its y axis across the $Q^2$ bins. This is
the only simulation-only figure saved to PDF.

In [ ]:
# (pairs, column, x range, x label, per_event, pair tag), one row each
MASS_GRID_ROWS = (
    (
        pippim_cut,
        "Mh",
        (0.28, 1.4),
        r"$M_{\pi^+\pi^-}$ [GeV]",
        True,
        r"$\pi^+ + \pi^-$",
    ),
    (
        pippi0_cut,
        "Mdiphoton",
        (0.0, 0.5),
        r"$M_{\gamma\gamma}$ [GeV]",
        True,
        r"$\gamma + \gamma$",
    ),
    (
        pippi0_window,
        "Mh",
        (0.28, 1.25),
        r"$M_{\pi^+\pi^0}$ [GeV]",
        True,
        r"$\pi^+ + \gamma + \gamma$",
    ),
)
q2_key, q2_edges, q2_label, q2_unit = MH_BINNINGS[0]  # the Q2 binning used above
n_cols = len(q2_edges) - 1

fig = plt.figure(figsize=(6 * n_cols, 4.5 * len(MASS_GRID_ROWS)))
grid = fig.add_gridspec(len(MASS_GRID_ROWS), n_cols, wspace=0.06, hspace=0.3)
for row, (pairs, key, xrange, xlabel, per_event, tag) in enumerate(MASS_GRID_ROWS):
    row_ax = None
    for col, (lo, hi) in enumerate(zip(q2_edges[:-1], q2_edges[1:])):
        ax = fig.add_subplot(grid[row, col], sharey=row_ax)
        row_ax = row_ax or ax
        in_bin = {
            label: dh[(dh[q2_key] >= lo) & (dh[q2_key] < hi)]
            for label, dh in pairs.items()
        }
        plot_mass_comparison(
            in_bin,
            n_events,
            key=key,
            bins=50,
            xrange=xrange,
            xlabel=xlabel,
            ylabel=r"pairs / (events $\cdot$ GeV)",
            per_event=per_event,
            # column headers on the top row only
            title=rf"${lo:g} < {q2_label} < {hi:g}${q2_unit}" if row == 0 else None,
            axes=(ax, None),
            **SIM_KWARGS,
        )
        ax.xaxis.label.set_size(16)
        ax.title.set_size(14)
        if col == 0:  # which particles are paired, top left of the row
            ax.text(
                0.05,
                0.92,
                tag,
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=16,
                color="grey",
            )
        if row or col:  # one legend, top-left
            ax.get_legend().remove()
        if col:
            ax.set_ylabel("")
            ax.tick_params(labelleft=False)

plt.savefig(f"{FIG_DIR}/mass_Q2_bins_grid.pdf", bbox_inches="tight")
plt.show()

## Comparison to data

The only figure that includes the RGA data: the $Q^2$ distribution of events passing the
DIS cuts ($Q^2 > 1$ GeV$^2$, $W > 2$ GeV, $y < 0.8$), and the SIDIS-selected
$\pi^+\pi^-$ and $\pi^+\pi^0$ ($\pi^0$ window) invariant masses, side by side. Each is
normalised to unit area, with Data filled light blue and a ratio-to-Data panel
underneath.

In [ ]:
def calculate_dis_vars(events, beam_e=BEAM_E):
    # Placeholder for the actual calculation of DIS variables
    # Replace the following line with the actual computation
    scat_e, has_e = scattered_electron(events)

    k = vector.obj(px=0.0, py=0.0, pz=beam_e, E=beam_e)  # beam
    P = vector.obj(px=0.0, py=0.0, pz=0.0, E=M_P)  # target
    kp = as_momentum(scat_e[has_e])
    q = k - kp

    Q2 = -q.tau2  # tau2 == m^2 for vector's 4-vectors
    nu = P.dot(q) / M_P
    x = Q2 / (2.0 * M_P * nu)
    y = nu / beam_e
    W = np.sqrt(np.maximum(M_P**2 + 2.0 * M_P * nu - Q2, 0.0))

    arrays = {
        "Q2": Q2.to_numpy(),
        "x": x.to_numpy(),
        "y": y.to_numpy(),
        "W": W.to_numpy(),
    }

    out = np.empty(
        len(arrays["Q2"]),
        dtype=[(name, values.dtype) for name, values in arrays.items()],
    )

    for name, values in arrays.items():
        out[name] = values
    return out

In [ ]:
events_dis = {
    "Full Sim": calculate_dis_vars(pflow_events),
    "Fast Sim": calculate_dis_vars(fs_events),
    "Data": calculate_dis_vars(data_events),
}
# Not named CUTS: the dihadron cut cell reads CUTS, and overwriting it here would
# silently change those cuts (sidis_mask defaults to M_X > 1.5) on a re-run.
DIS_CUTS = dict(q2_min=1.0, w_min=2.0, y_max=0.8)
DIS_mask = {}
events_dis_cuts = {}
for name, col in events_dis.items():
    DIS_mask[name] = (
        (col["Q2"] > DIS_CUTS["q2_min"])
        & (col["W"] > DIS_CUTS["w_min"])
        & (col["y"] < DIS_CUTS["y_max"])
    )
    events_dis_cuts[name] = col[DIS_mask[name]]

In [ ]:
style_dict = {
    "Data": {
        "label": "CLAS12 RGA Data",  # legend only; ratios stay "/Data"
        "linewidth": 2,
        "color": "#1f77b4",
        "alpha": 0.3,
        "histtype": "stepfilled",
    },
    "Full Sim": {"linewidth": 2, "color": "black", "alpha": 1, "histtype": "step"},
    "Fast Sim": {"linewidth": 2, "color": "crimson", "histtype": "step"},
}

## Angular-distribution helpers

`select_particles` / `particles_by_label` build per-species samples for Data, Full Sim and
Fast Sim, with `theta` and `phi` rebuilt from the four-vectors so the same code path
serves all three samples despite the eval / data format difference. Nothing is drawn
from them here; hand `particles_by_label(PDG_ID, dis_mask=DIS_mask)` to
`plot_mass_comparison` for a quick check.

In [ ]:
# ----------------------------------------------------------------------
# per-particle angular distributions for a single species
# ----------------------------------------------------------------------
PDG_NAMES = {
    11: r"$e^-$",
    -11: r"$e^+$",
    22: r"$\gamma$",
    45: r"$d$",
    211: r"$\pi^+$",
    -211: r"$\pi^-$",
    321: r"$K^+$",
    -321: r"$K^-$",
    2112: r"$n$",
    2212: r"$p$",
    -2212: r"$\bar{p}$",
}


def select_particles(events, pdg_id, degrees=True):
    """Flatten every ``pdg_id`` particle in ``events`` into one row per particle.

    Returns a flat record array with the same convention as the dihadron
    builders (theta / phi / pt / p / eta / E per particle), so it can be handed
    straight to ``plot_mass_comparison``. Angles are in degrees by default,
    which is the natural scale for the CLAS12 forward acceptance.
    """
    sel = as_momentum(events[events.pdgId == pdg_id])
    to_deg = np.degrees if degrees else (lambda v: v)
    out = {
        "theta": to_deg(sel.theta),
        "phi": to_deg(sel.phi),
        "pt": sel.pt,
        "p": sel.p,
        "eta": sel.eta,
        "E": sel.E,
    }
    return ak.Array({name: ak.flatten(col, axis=1) for name, col in out.items()})


def dis_event_mask(events, dis_mask):
    """Expand a DIS mask onto every event in ``events``.

    ``calculate_dis_vars`` compresses to events *with* a scattered electron
    (``scat_e[has_e]``), so ``DIS_mask`` is shorter than ``events``. Scatter it
    back into the ``has_e`` slots, in the same order it was built.
    """
    has_e = ak.to_numpy(scattered_electron(events)[1])
    if len(dis_mask) != has_e.sum():
        raise ValueError(
            f"DIS mask has {len(dis_mask):,} entries but {has_e.sum():,} events have "
            "a scattered electron -- re-run the DIS cell after changing N_EVENTS"
        )
    keep = np.zeros(len(has_e), dtype=bool)
    keep[has_e] = dis_mask
    return keep


def particles_by_label(pdg_id, dis_mask=None, **kwargs):
    """Data / Full Sim / Fast Sim selections for one pdgId.

    Pass ``dis_mask=DIS_mask`` to keep only events passing the event-level DIS
    cuts. Returns ``(particles, n_kept)``; ``n_kept`` is the surviving event
    count per sample, which is what ``per_event`` normalisation needs.
    """
    events_by_label = {
        "Data": data_events,
        "Full Sim": pflow_events,
        "Fast Sim": fs_events,
    }
    particles, n_kept = {}, {}
    for label, events in events_by_label.items():
        if dis_mask is not None:
            events = events[dis_event_mask(events, dis_mask[label])]
        particles[label] = select_particles(events, pdg_id, **kwargs)
        n_kept[label] = len(events)
    return particles, n_kept

In [ ]:
# ----------------------------------------------------------------------
# combined data-comparison figure: Q2, pi+pi- Mh and pi+pi0 Mh side by side
# ----------------------------------------------------------------------
# Uses events_dis_cuts (DIS cell above) and pippim_cut / pippi0_window
# (invariant-mass section). make_panel builds each (main, ratio) pair;
# transplant redraws an existing Axes into another figure (matplotlib axes
# cannot be moved between figures) and is kept for re-composing panels.
from matplotlib.collections import LineCollection, PolyCollection
from matplotlib.patches import PathPatch


def transplant(src, dst, fontsize=None):
    """Redraw every artist of Axes ``src`` onto Axes ``dst``."""
    for line in src.lines:
        x, y = line.get_data()
        props = dict(
            color=line.get_color(), lw=line.get_linewidth(), ls=line.get_linestyle()
        )
        # axhline / axvline live in blended coords: 2 points, companion == (0, 1)
        if len(x) == 2 and x[0] == x[1] and tuple(y) == (0.0, 1.0):
            dst.axvline(x[0], **props)
        elif len(y) == 2 and y[0] == y[1] and tuple(x) == (0.0, 1.0):
            dst.axhline(y[0], **props)
        else:
            dst.plot(
                x,
                y,
                marker=line.get_marker(),
                ms=line.get_markersize(),
                drawstyle=line.get_drawstyle(),
                label=line.get_label(),
                **props,
            )

    for coll in src.collections:
        if isinstance(coll, LineCollection):  # errorbar bars
            # empty segments come from NaN ratio bins and cannot be re-added
            segments = [seg for seg in coll.get_segments() if len(seg)]
            if segments:
                dst.add_collection(
                    LineCollection(
                        segments,
                        colors=coll.get_colors(),
                        linewidths=coll.get_linewidths(),
                    )
                )
        elif isinstance(coll, PolyCollection):  # fill_between band
            for path in coll.get_paths():
                dst.add_patch(PathPatch(path, facecolor=coll.get_facecolor()[0], lw=0))

    dst.set_xlim(src.get_xlim())
    dst.set_ylim(src.get_ylim())
    dst.set_yscale(src.get_yscale())
    dst.set_xlabel(src.get_xlabel(), fontsize=fontsize)
    dst.set_ylabel(src.get_ylabel(), fontsize=fontsize)
    dst.set_title(src.get_title(), fontsize=fontsize)
    if fontsize is not None:
        dst.tick_params(labelsize=fontsize)


def make_panel(fig, spec, sharex_from=None, n_ratio=1):
    """A main axes with ``n_ratio`` ratio axes below it, inside one gridspec slot."""
    sub = spec.subgridspec(
        1 + n_ratio, 1, height_ratios=[3] + [1] * n_ratio, hspace=0.05
    )
    axes = [fig.add_subplot(sub[0])]
    axes += [fig.add_subplot(sub[k], sharex=axes[0]) for k in range(1, 1 + n_ratio)]
    for a in axes[:-1]:
        a.tick_params(labelbottom=False)
    return axes


DATA_COMP_PLOTS = (
    # (samples, column, x range, x label, title)
    (events_dis_cuts, "Q2", (1.0, 7.0), r"$Q^2$ [GeV$^2$]", r"$Q^2$ distribution"),
    (
        pippim_cut,
        "Mh",
        (0.28, 1.4),
        r"$M_{\pi^+\pi^-}$ [GeV]",
        r"$\pi^+\pi^-$ invariant mass",
    ),
    (
        pippi0_window,
        "Mh",
        (0.28, 1.0),
        r"$M_{\pi^+\pi^0}$ [GeV]",
        r"$\pi^+\pi^0$ invariant mass",
    ),
)

# 6 in tall before the Fast/Full row: 7.56 keeps every other panel its size
fig = plt.figure(figsize=(6.5 * len(DATA_COMP_PLOTS), 7.56))
outer = fig.add_gridspec(1, len(DATA_COMP_PLOTS), wspace=0.25)
for i, (spec, (samples, key, xrange, xlabel, title)) in enumerate(
    zip(outer, DATA_COMP_PLOTS)
):
    ax, rax, rax2 = make_panel(fig, spec, n_ratio=2)
    # unit-area normalisation, so n_events is not used
    plot_mass_comparison(
        samples,
        n_events,
        key=key,
        bins=75,
        xrange=xrange,
        xlabel=xlabel,
        title=title,
        per_event=False,
        ratio_ylim=(0.8, 1.2),
        axes=(ax, rax),
        # fast-sim fidelity next to the full-sim mismodelling of the data
        extra_ratios=(rax2, [("Fast Sim", "Full Sim"), ("Full Sim", "Data")]),
    )
    if i:  # one set of legends, on the left-most panel
        ax.get_legend().remove()
        rax.get_legend().remove()
        rax2.get_legend().remove()

plt.savefig(f"{FIG_DIR}/combined_data_comp.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# diphoton invariant mass in pi+ gamma gamma pairs: Data vs Full Sim vs Fast Sim,
# unit-area normalised like the combined data-comparison figure above
fig, ax, rax = plot_mass_comparison(
    pippi0_cut,
    n_events,
    key="Mdiphoton",
    bins=75,
    xrange=(0.0, 0.5),
    xlabel=r"$M_{\gamma\gamma}$ [GeV]",
    title=r"diphoton invariant mass in $\pi^+\gamma\gamma$ pairs",
    per_event=True,
    ratio_ylim=(0.8, 1.2),
)
ax.axvline(M_PI0, color="grey", lw=0.8, ls=":")
ax.annotate(r"$\pi^0$", (M_PI0, 0.92), xycoords=("data", "axes fraction"), color="grey")
plt.show()